# ATML PA2 - Task 1 (DPO) orchestration on Kaggle (2x T4)

This notebook only **orchestrates** the repository scripts; all experiment code lives in `task1_dpo/`.

**Kaggle settings:** Accelerator = *GPU T4 x2*, Internet = *On*. Use **Save Version -> Save & Run All (Commit)** so the
~2-3 h run continues without the browser open. Everything is written to `/kaggle/working/` and the final cell zips the results.

What runs (see `task1_dpo/run_all.sh`):

| GPU 0 | GPU 1 |
|---|---|
| standard DPO (1 epoch) -> eval | beta=0.03 fork -> eval |
| SFT baseline eval | beta=0.10 fork -> eval |
| length-balanced DPO (1 epoch) -> eval -> length analysis | beta=0.30 fork -> eval |

then `task1_dpo.make_report` builds `report/task1/` (tables, figures, qualitative candidates).

## 1. Get the code
Set `REPO_URL` to **your** GitHub repository (the one containing the Task 1 implementation). For a private repo, add a Kaggle secret `GITHUB_TOKEN`.

In [ ]:
REPO_URL = "https://github.com/alihasnaink/atml-pa2.git"   # <-- edit
BRANCH = "main"
WORKDIR = "/kaggle/working/repo"

import os, subprocess
url = REPO_URL
try:  # optional: private repository
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass
if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, WORKDIR], check=True)
os.chdir(WORKDIR)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

## 2. Install the pinned environment

In [ ]:
!pip install -q -r requirements.txt
!python -m scripts.check_environment
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## 3. Download and validate the fixed course assets

In [ ]:
!python -m scripts.download_assets
!python -m scripts.validate_assets

## 4. Run all Task 1 experiments
Resumable: if the session restarts, re-run this cell and finished adapters/evaluations are skipped. Per-GPU logs: `results/task1_dpo/logs/`.

In [ ]:
!bash task1_dpo/run_all.sh

## 5. Quick look at the results

In [ ]:
from IPython.display import Markdown, Image, display
for t in ["dpo_summary", "length_strata", "word_limit", "paired_generation_differences", "dataset_length_stats"]:
    p = f"report/task1/tables/{t}.md"
    if os.path.exists(p):
        display(Markdown(open(p).read()))
for f in ["beta_sweep", "training_curves", "length_strata_accuracy", "margin_vs_length_difference", "generated_length_and_word_limit"]:
    p = f"report/task1/figures/{f}.png"
    if os.path.exists(p):
        display(Image(p))

## 6. Package the artifacts
Download `task1_artifacts.zip` from the Output tab and unzip it at the repository root. It contains `results/task1_dpo/`, `report/task1/` and the LoRA adapters in `outputs/task1_dpo/` (adapters are git-ignored; keep `outputs/task1_dpo/standard` because Task 4 uses it).

In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/task1_artifacts.zip results/task1_dpo report/task1 outputs/task1_dpo
!ls -lh /kaggle/working/task1_artifacts.zip